# BACI HS92 — Network Analysis

Loads `baci_edges_country.parquet` produced by `baci_analysis.ipynb` and computes network statistics over time.

**Input:** `baci_edges_country.parquet` — country-level trade flows  
**Units:** edge `value` = USD billions

## Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

In [ ]:
# ── Same path as in baci_analysis.ipynb ──────────────────────────────────
DATA_BACI = Path("/Users/olivia_peter/Masters BSE/Term 3/Thesis BSE/Code Thesis/BACI_HS92_V202601")  # <-- edit this
OUT_BACI  = DATA_BACI

assert DATA_BACI.exists(), f"DATA_BACI not found: {DATA_BACI}"
print("OUT_BACI:", OUT_BACI)

## Step 4 — Core Network Parameters Over Time

In [ ]:
# ── Load country-level edge list ─────────────────────────────────────────
edges_df = pd.read_parquet(OUT_BACI / "baci_edges_country.parquet")

print(f"Loaded baci_edges_country.parquet: {edges_df.shape}")
print(f"Years covered: {sorted(edges_df['year'].unique())}")
edges_df.head()

In [ ]:
# ── Helper: build weighted DiGraph for one year ───────────────────────────
def build_digraph(year_df: pd.DataFrame) -> nx.DiGraph:
    return nx.from_pandas_edgelist(
        year_df,
        source="source",
        target="target",
        edge_attr="value",
        create_using=nx.DiGraph(),
    )

In [ ]:
# ── Compute global network metrics per year ───────────────────────────────
global_records = []
prev_edge_set  = None

for year, year_df in edges_df.groupby("year"):

    G = build_digraph(year_df)
    n = G.number_of_nodes()
    m = G.number_of_edges()

    # Density
    density = nx.density(G)

    # Largest weakly connected component — diameter
    G_und = G.to_undirected()
    wccs  = list(nx.connected_components(G_und))
    largest_wcc = max(wccs, key=len)
    wcc_frac    = len(largest_wcc) / n
    G_wcc       = G_und.subgraph(largest_wcc)
    diameter    = nx.diameter(G_wcc)

    # Average weighted clustering coefficient
    avg_clust = nx.average_clustering(G, weight="value")

    # Average in/out-strength (USD billions)
    in_str  = dict(G.in_degree(weight="value"))
    out_str = dict(G.out_degree(weight="value"))
    avg_in  = np.mean(list(in_str.values()))
    avg_out = np.mean(list(out_str.values()))

    # Jaccard edge-set similarity vs previous year
    curr_edge_set = set(G.edges())
    if prev_edge_set is not None:
        inter   = len(curr_edge_set & prev_edge_set)
        union   = len(curr_edge_set | prev_edge_set)
        jaccard = inter / union if union > 0 else np.nan
    else:
        jaccard = np.nan
    prev_edge_set = curr_edge_set

    # Out-strength Herfindahl index (export concentration)
    out_vals = np.array(list(out_str.values()), dtype=float)
    total    = out_vals.sum()
    shares   = out_vals / total if total > 0 else out_vals
    hhi      = float(np.sum(shares ** 2))

    global_records.append({
        "year":                      year,
        "n_nodes":                   n,
        "n_edges":                   m,
        "density":                   density,
        "wcc_largest_frac":          wcc_frac,
        "diameter_largest_wcc":      diameter,
        "avg_weighted_clustering":   avg_clust,
        "avg_in_strength_bUSD":      avg_in,
        "avg_out_strength_bUSD":     avg_out,
        "jaccard_vs_prev_year":      jaccard,
        "hhi_out_strength":          hhi,
    })

    print(
        f"{year} | nodes {n:3d} | edges {m:5d} | density {density:.4f} | "
        f"diam {diameter:2d} | wcc {wcc_frac:.3f} | HHI {hhi:.4f} | "
        f"Jac {jaccard:.4f}" if not np.isnan(jaccard) else
        f"{year} | nodes {n:3d} | edges {m:5d} | density {density:.4f} | "
        f"diam {diameter:2d} | wcc {wcc_frac:.3f} | HHI {hhi:.4f} | Jac  n/a"
    )

In [ ]:
# ── Save and display global network stats ─────────────────────────────────
network_stats = pd.DataFrame(global_records)

stats_path = OUT_BACI / "network_stats.csv"
network_stats.to_csv(stats_path, index=False, float_format="%.6f")
print(f"Saved \u2192 {stats_path.name}")
print()

display_cols = {
    "year":                    "Year",
    "n_nodes":                 "Nodes",
    "n_edges":                 "Edges",
    "density":                 "Density",
    "wcc_largest_frac":        "WCC frac",
    "diameter_largest_wcc":    "Diameter",
    "avg_weighted_clustering": "Avg W-Clust",
    "avg_in_strength_bUSD":    "Avg In-Str ($B)",
    "avg_out_strength_bUSD":   "Avg Out-Str ($B)",
    "jaccard_vs_prev_year":    "Jaccard",
    "hhi_out_strength":        "HHI",
}
network_stats.rename(columns=display_cols)[list(display_cols.values())]

In [ ]:
# ── Compute CHN and USA node metrics per year ─────────────────────────────
TARGET_NODES = ["CHN", "USA"]
node_records = []

for year, year_df in edges_df.groupby("year"):
    G  = build_digraph(year_df)
    pr = nx.pagerank(G, weight="value")

    for node in TARGET_NODES:
        if node not in G:
            print(f"  Warning: {node} not in graph for {year}")
            continue

        node_records.append({
            "year":              year,
            "country":           node,
            "in_degree":         G.in_degree(node),
            "out_degree":        G.out_degree(node),
            "in_strength_bUSD":  G.in_degree(node,  weight="value"),
            "out_strength_bUSD": G.out_degree(node, weight="value"),
            "pagerank":          pr.get(node, np.nan),
        })

chn_usa_stats = pd.DataFrame(node_records)

node_stats_path = OUT_BACI / "chn_usa_stats.csv"
chn_usa_stats.to_csv(node_stats_path, index=False, float_format="%.6f")
print(f"Saved \u2192 {node_stats_path.name}")
print()
chn_usa_stats